# 00 - Setup & Earth Engine authentication

**Colombo UHI practicum - Phase 0.** Run this notebook top-to-bottom in
**Google Colab** once per fresh runtime. Cells marked `# COLAB: RUN THIS CELL`
require *your* Google account and cannot be run by anyone else.

Prerequisites:

1. This repository pushed to **your GitHub** (see README).
2. A Google Cloud project **registered for Earth Engine** (free,
   noncommercial): <https://code.earthengine.google.com/register>

Rules of the repo: notebooks only orchestrate - all logic lives in
`src/colombo_uhi/`, and every constant comes from `config/params.yaml`.


In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only


In [ ]:
# COLAB: RUN THIS CELL
# ONE pip call for the whole requirement set (requirements.txt explains why).
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")


In [ ]:
# COLAB: RUN THIS CELL
# Report library versions (paste this output into your report appendix).
import sys
from importlib import metadata

print("Python:", sys.version.split()[0])
packages = [
    "earthengine-api", "geemap", "geopandas", "rasterio", "xarray",
    "rioxarray", "numpy", "pandas", "scipy", "statsmodels", "scikit-learn",
    "pymannkendall", "libpysal", "esda", "spreg", "mgwr", "matplotlib",
    "seaborn", "PyYAML", "tqdm",
]
for pkg in packages:
    try:
        print(f"{pkg:>15}  {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print(f"{pkg:>15}  !! MISSING")


In [ ]:
# COLAB: RUN THIS CELL
# Make src/ importable and load config/params.yaml (single source of truth).
import os
import sys

sys.path.insert(0, os.path.abspath("src"))
from colombo_uhi import __version__, load_params

params = load_params()
print("colombo_uhi package:", __version__)
print("Study period:", params["time"]["start_year"], "-", params["time"]["end_year"])
print("CRS / grid:", params["crs"]["analysis_epsg"], "/", params["crs"]["analysis_scale_m"], "m")
print("EE project id in params.yaml:", params["gcp"]["ee_project_id"])


In [ ]:
# COLAB: RUN THIS CELL
# First run in a fresh VM opens a Google authentication prompt - approve it.
# Project id comes from config/params.yaml (gcp.ee_project_id).
# Per-session override if ever needed: os.environ["EE_PROJECT"] = "..."
# or init_ee(project_id="...", force=True).

from colombo_uhi.auth import init_ee

project = init_ee()
print("Earth Engine initialised with project:", project)


In [ ]:
# COLAB: RUN THIS CELL
# Trivial Earth Engine calls proving the session works.
from colombo_uhi.auth import ee_smoke_test

result = ee_smoke_test(params)
print(result)
assert result["ok"], "Smoke test failed - inspect the values above."
print()
print("Earth Engine session OK - setup complete.")
print("Expected: one_plus_one == 2 and srtm_bands == ['elevation'].")


## Troubleshooting

| Symptom | Fix |
|---|---|
| pip says *restart the runtime* | Runtime > Restart session, then re-run from the **clone cell** (skip pip). |
| `No Earth Engine Cloud project id configured` | Check `gcp.ee_project_id` in `config/params.yaml` (should be `research-uhi-484404`) and re-run the auth cell. |
| `ee.Initialize failed for project ...` | The project is not EE-registered or the EE API is off - fix at <https://code.earthengine.google.com/register>, wait ~1 min, re-run with `init_ee(force=True)`. |
| Auth prompt loops / wrong account | Runtime > Disconnect and delete runtime, then rerun; pick the Google account that owns the EE project. |
| `params file not found` | You are not in the repo dir - re-run the clone cell (it `%cd`s there). |

Fresh VMs always re-prompt once - that is normal. Within one runtime,
`init_ee()` never re-prompts (idempotent).

**Next:** `01_aoi_and_boundaries.ipynb` (Phase 1) - not written yet.
